In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split

from model import GATClassifier

# -----------------------------
# SETTINGS
# -----------------------------
BATCH_SIZE = 64
NUM_EPOCHS = 1
LR = 0.001
HIDDEN_CHANNELS = 64
NUM_CLASSES = 4  # Alzheimer's classes
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# -----------------------------
# LOAD DATA
# -----------------------------
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

# -----------------------------
# INITIALIZE MODEL
# -----------------------------
model = GATClassifier(
    in_channels=1,              # Feature: mean intensity
    hidden_channels=HIDDEN_CHANNELS,
    out_channels=NUM_CLASSES
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = torch.nn.CrossEntropyLoss()

# -----------------------------
# TRAINING LOOP
# -----------------------------
def train():
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in train_loader:
        batch = batch.to(DEVICE)
        optimizer.zero_grad()

        out = model(batch.x, batch.edge_index, batch.batch)
        loss = loss_fn(out, batch.y.view(-1))
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = out.argmax(dim=1)
        correct += (preds == batch.y.view(-1)).sum().item()
        total += batch.num_graphs

    acc = correct / total
    return total_loss / len(train_loader), acc

# -----------------------------
# VALIDATION LOOP
# -----------------------------
def evaluate():
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in val_loader:
            batch = batch.to(DEVICE)
            out = model(batch.x, batch.edge_index, batch.batch)
            preds = out.argmax(dim=1)
            correct += (preds == batch.y.view(-1)).sum().item()
            total += batch.num_graphs

    acc = correct / total
    return acc

# -----------------------------
# RUN TRAINING
# -----------------------------
print("Training model...\n")
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train()
    val_acc = evaluate()
    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

print("\n✅ Training complete.")


In [ ]:
""" import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Parameter
from torch_geometric.nn import GATConv, global_mean_pool


class BGANLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(BGANLayer, self).__init__()
        self.gat = GATConv(in_channels, out_channels, heads=1, concat=True)
        self.local_weight = Parameter(torch.randn(out_channels, out_channels))

    def forward(self, x, edge_index):
        # Apply graph attention
        x_att = self.gat(x, edge_index)
        # Local attention update
        out = F.relu(torch.matmul(x_att, self.local_weight) + x_att)
        return out


class GlobalAttention(nn.Module):
    def __init__(self, in_channels):
        super(GlobalAttention, self).__init__()
        self.linear = nn.Linear(in_channels, 1)

    def forward(self, x, batch):
        # x: [N_nodes, F]; batch: [N_nodes] with graph ids
        weights = torch.sigmoid(self.linear(x))
        x = x * weights
        x_pool = global_mean_pool(x, batch)  # [N_graphs, F]
        return x_pool


class BGANClassifier(nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes):
        super(BGANClassifier, self).__init__()
        self.local_layer = BGANLayer(in_channels, hidden_channels)
        self.global_attention = GlobalAttention(hidden_channels)
        self.classifier = nn.Linear(hidden_channels, num_classes)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.local_layer(x, edge_index)          # Local graph attention
        x = self.global_attention(x, batch)          # Global pooling with attention
        out = self.classifier(x)                     # Graph-level classification
        return out
 """

In [ ]:
""" import dgl
import numpy as np
import torch as th
import torch.nn as nn
import torch.nn.functional as F
import dgl.nn.pytorch as dglnn
import random
from torch.nn.parameter import Parameter
device=th.device('cuda' if th.cuda.is_available() else 'cpu')

class BGAN(nn.Module):
    def __init__(self,in_dim, out_dim, n_classes):
        super(BGAN, self).__init__()
        self.in_dim = in_dim
        self.classify = nn.Linear(out_dim, n_classes)
        self.GATLayer = GATLayer(in_dim,out_dim)
        self.conv = dglnn.GraphConv(in_dim, 1)
        self.localw= Parameter(th.randn(self.max_neighs+self.in_dim-1,out_dim))
        self.ReLU = nn.ReLU(inplace=True)
        self.Sigmoid = nn.Sigmoid()
        self.Softmax = nn.Softmax(dim=1)
    
    def reset_parameters(self):
        gain = nn.init.calculate_gain('relu')
        nn.init.xavier_normal_(self.localw, gain=gain)

    def get_graphs(self, block):
        g = block
        self.max_neighs = 10

        in_edges = th.tensor([]).to(device)
        out_edges = th.tensor([]).to(device)
        for node in range(len(g.dstnodes())): 
            src, dst = g.in_edges(int(node))
            if src.shape[0] == 0:
                src = th.tensor([int(node)], dtype=th.int64)
            src = src.repeat(self.max_neighs)[:self.max_neighs]
            dst = dst.repeat(self.max_neighs)[:self.max_neighs]
            out_edges = th.cat([out_edges,src]).to(device)
            in_edges = th.cat([in_edges,dst]).to(device)
        graph = dgl.graph((out_edges.long(),in_edges.long())).to(device)
        return graph

    def GlobalAttention(self,g,h):
        globalweight = self.Softmax(self.conv(g,h).reshape(self.conv(g,h).shape[0]//self.in_dim,self.in_dim))
        globalweight = globalweight.reshape(globalweight.shape[0]*self.in_dim,1)
        return globalweight

    def LocalAttention(self,g,h):
        graph = self.get_graphs(g)
        output = self.GATLayer(graph,h)
        updatefeat = self.ReLU(th.mm(output,self.localw) + h)
        return updatefeat
    
    def forward(self,g,h):
        '''Local Attention: Update features layers'''
        feats = self.LocalAttention(g,h)   
        '''Global Attention: Calculate weight layers'''
        weight = self.GlobalAttention(g,h)  
        
        '''Classifier'''
        updatafeat = weight * feats                       
        with g.local_scope(): 
            g.ndata['h'] = updatafeat
            hg = dgl.mean_nodes(g, 'h')
        return self.classify(hg)

class GATLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super(GATLayer, self).__init__()
        self.fc = nn.Linear(in_dim, out_dim, bias=False)
        self.attn_fc = nn.Linear(2 * out_dim, 1, bias=False)
        self.conv = CNN()
        self.reset_parameters()
        
    def reset_parameters(self):
        gain = nn.init.calculate_gain('relu')
        nn.init.xavier_normal_(self.fc.weight, gain=gain)
        nn.init.xavier_normal_(self.attn_fc.weight, gain=gain)

    def edge_attention(self, edges):
        z2 = th.cat([edges.src['z'], edges.dst['z']], dim=1)
        a = self.attn_fc(z2)
        return {'e': F.leaky_relu(a)}
    def message_func(self, edges):
        return {'z': edges.src['z'], 'e': edges.data['e']}
    def reduce_func(self, nodes):
        alpha = F.softmax(nodes.mailbox['e'], dim=1)
        h = self.conv(alpha * nodes.mailbox['z'])
        return {'h': h}
    def forward(self, g,h):
        z = self.fc(h)
        g.ndata['z'] = z
        g.apply_edges(self.edge_attention)
        g.update_all(self.message_func, self.reduce_func)
        return g.ndata.pop('h')

class CNN(th.nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        self.convrow = th.nn.Sequential(
            th.nn.Conv2d(in_channels=1,
                        out_channels=1,
                        kernel_size=(2,self.in_dim)),           
            th.nn.BatchNorm2d(num_features=1),
            th.nn.ReLU(),
        )
        self.convcol = th.nn.Sequential(
            th.nn.Conv2d(in_channels=1,
                         out_channels=1,
                         kernel_size=(self.max_neighs, 1)),
            th.nn.BatchNorm2d(num_features=1),
            th.nn.ReLU(),                                
        )
    def reset_parameters(self):
        gain = nn.init.calculate_gain('relu')
        nn.init.xavier_normal_(self.convrow, gain=gain)
        nn.init.xavier_normal_(self.convcol, gain=gain)

    def forward(self, feats):
        feats = feats.unsqueeze(1)              
        featsrow = self.convrow(feats)          
        featsrow = np.squeeze(featsrow)      

        featscol = self.convcol(feats)        
        featscol = np.squeeze(featscol)      
       
        feats = th.cat((featsrow,featscol),dim=1)    
        return feats """

In [ ]:
from dataset import create_dataframe, stratified_patient_split, MRISeqDataset, custom_collate

directory = './Data'
df = create_dataframe(directory)
df_train, df_test = stratified_patient_split(df, test_size=0.25)

train_dataset = MRISeqDataset(df_train, directory)
test_dataset = MRISeqDataset(df_test, directory)

In [ ]:
from torch.utils.data import DataLoader

BATCH_SIZE = 64
NUM_EPOCHS = 10
LEARNING_RATE = 0.001
HIDDEN_CHANNELS = 64
NUM_CLASSES = 2

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=custom_collate)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=custom_collate)

In [36]:
sequences, labels = next(iter(train_loader))
# sequences: List[List[Data]]
# labels: Tensor of shape [batch_size]

for seqs, labels in train_loader:
    print("Batch of scans:", len(seqs))
    print("Scan 0 has", len(seqs[0]), "slices")
    break


New edge construction: 0.004563808441162109
New edge construction: 0.003866910934448242
New edge construction: 0.005007028579711914
New edge construction: 0.004389286041259766
New edge construction: 0.004946231842041016
New edge construction: 0.00444793701171875
New edge construction: 0.0059452056884765625
New edge construction: 0.004734516143798828
New edge construction: 0.007197141647338867
New edge construction: 0.005227804183959961
New edge construction: 0.005090951919555664
New edge construction: 0.004038095474243164
New edge construction: 0.003722667694091797
New edge construction: 0.004797697067260742
New edge construction: 0.00395655632019043
New edge construction: 0.004663228988647461
New edge construction: 0.005248069763183594
New edge construction: 0.0047435760498046875
New edge construction: 0.006389141082763672
New edge construction: 0.004632472991943359
New edge construction: 0.004488945007324219
New edge construction: 0.005257844924926758
New edge construction: 0.0044450